In [3]:
import os
import glob
import polars as pl
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Feature Data Directory (Where the CSVs to be updated are located)
#FEAT_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Inferno")

FEAT_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Inferno")

# Raw Data Directory (To extract the 'round_end' ground truth)
RAW_DATA_DIR = Path(r"F:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Standardized Team Mapping
TEAM_MAP = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}

# ==========================================
# 2. DATA RETRIEVAL & UPDATING LOGIC
# ==========================================
def update_features_with_targets():
    """Iterates through feature CSVs, retrieves ground-truth labels, and appends them."""
    
    if not FEAT_DIR.exists():
        print(f"Error: Feature directory does not exist -> {FEAT_DIR}")
        return

    csv_files = list(FEAT_DIR.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in -> {FEAT_DIR}")
        return

    print(f"Found {len(csv_files)} matches to update. Processing...\n")
    updated_count = 0

    for filepath in csv_files:
        match_id = filepath.stem
        
        # --- A. Find the Raw Round_End Data ---
        match_dirs = list(RAW_DATA_DIR.rglob(match_id))
        if not match_dirs:
            print(f"Skipping {match_id}: Raw data directory not found.")
            continue
            
        re_files = glob.glob(str(match_dirs[0] / "round_end*"))
        if not re_files:
            print(f"Skipping {match_id}: 'round_end' channel missing.")
            continue

        # --- B. Extract the Target Labels ---
        try:
            df_re = pl.read_parquet(re_files[0])
            
            # Map the winner codes and keep only 'round' and 'target'
            df_labels = df_re.with_columns(
                pl.col("winner_team_code").cast(pl.Utf8).replace(TEAM_MAP).alias("target")
            ).select(["round", "target"])
            
        except Exception as e:
            print(f"Error reading 'round_end' for {match_id}: {e}")
            continue

        # --- C. Update the Feature Data ---
        try:
            df_feat = pl.read_csv(filepath)
            
            # Safety check: If "target" already exists from a previous run, drop it to prevent duplicates like "target_right"
            if "target" in df_feat.columns:
                df_feat = df_feat.drop("target")
                
            # Perform a Left Join on the "round" column
            # This ensures every round in the feature data gets its corresponding label
            df_updated = df_feat.join(df_labels, on="round", how="left")
            
            # --- D. Save the Changes (Overwrite) ---
            df_updated.write_csv(filepath)
            updated_count += 1
            
            print(f"Successfully updated: {match_id}")
            
        except Exception as e:
            print(f"Error updating feature file {match_id}: {e}")

    print("\n" + "="*40)
    print(f" UPDATE COMPLETE: {updated_count}/{len(csv_files)} files modified.")
    print("="*40)

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Note: Because this script overwrites data, ensure no other processes are locking the CSV files.
    update_features_with_targets()

Found 69 matches to update. Processing...

Successfully updated: 985470e7-2548-41ba-8e50-2c4fbd9f13eb
Successfully updated: b2637562-2abd-4492-a9bd-0d29fa93cf55
Successfully updated: 268a8b47-9ffc-4d71-97ea-942442d8e88b
Successfully updated: 687fab70-73f6-4f5a-9fbf-d7a13cb0be30
Successfully updated: 60d0df56-8955-4bdd-8589-0050acefb993
Successfully updated: 4bbe0d2e-c424-42ea-b919-d6a56173cdea
Successfully updated: dfec78c3-39fa-476e-8868-524a57c12267
Successfully updated: 9a864b58-fb4e-4927-a9cb-c7570e2cff39
Successfully updated: b765a810-96d6-4fde-ae74-0761bf52748a
Successfully updated: 4594283b-27e8-48e2-a030-81823589e628
Successfully updated: 7ea99526-f861-4cf5-8ec6-8e3575c09f9f
Successfully updated: 9cfa5958-6198-43a1-a3a1-e86559131db8
Successfully updated: 3c133f39-7f17-49cb-b8bb-816ef47ff4ef
Successfully updated: d42b8f44-ca78-4729-82bb-53bb0af520b2
Successfully updated: 2c263e4d-080c-402a-8ffc-0e7db32042d3
Successfully updated: 424eb3e3-fab3-42df-beb3-b9c88eaddf68
Successfully 